#### Security in FastAPI

Security is an esstenial aspect of any api, especially when working with sensitive data or user authentication, In FastAPI various tools and strategies are aviable to help secure your api,

this section will explore

1. how to handle CORS (Cross-Origin Resource sharing),

2. implementing authentication using OAuth2

3. protect sensitive data in the FastAPI applications

In [5]:
'''
CORS is a mechanism, that allows web browsers to make requests to a domain other than the one that served the orignal web pahe
this is common situation for client-side java script, which runs on a borwser and needs to interact with a server hosted on a different domain
by default fastapi blocks cross-origin requests for security reasons however in many cases
youll need to enable cors to allow frontend applications to interact with your backend

allow_origins: list defines which domains can send the requests to your api
allow_methods: defines http methods are allowed from the client
allow_headers: ensure that client can send custom headers if needed


'''
from fastapi import FastAPI
from starlette.middleware.cors import CORSMiddleware

app = FastAPI()

#allow cross origin request for specific origins
origins = [
    "https://example.com",
    "http:localhost.com",
]

#add cors middleware to the app
app.add_middleware(
    CORSMiddleware,
    allow_orgins = origins,
    allow_credentials = True,#allow cookies to be sent
    allow_methods = ['GET','POST','PUT','DELETE'],
    allow_headers=["*"]
)


In [6]:
'''

authentication with OAuth 2

Authentication is the process of verifying the identity of a user or client, Oauth
OAuth2 is a popular standard for authorization, which allows third party applications to access resources
on behalf of a user

Fastapi supports OAuth2 and can integrate seamlessly with tools such as JWT (Json Web Tokens) to implement token based authentication

In this example well demostrate how to implement OAuth2 for authentication using the password flow
where user provides their credentials (username and password) and in exchange they recieve a JWT token
'''

from pydantic import BaseModel

# Define pydantic model for user authentication
class User(BaseModel):
  username : str
  password: str

#model to represent the token recieved after authentication
class Token(BaseModel):
  access_token:str
  token_type:str



In [7]:
!pip install passlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 9.6 MB/s eta 0:00:00


In [ ]:
from typing_extensions import deprecated
#create functions for password authentication

from fastapi import Depends,HTTPException
from fastapi.security import OAuth2PasswordBearer
from passlib.context import CryptContext
from datetime import datetime,timedelta
import jwt

#OAuth2PasswordBearer is used to extract the token from the request header
'''

comple jwt authentication flow

1. LOGIN

Frontend  -------- username/password -------->  Backend
Frontend  <----------- JWT Token -------------  Backend

2. ACCESSING PROTECTED ROUTES

Frontend -------- JWT Token --------> Backend

                 LOGIN

Frontend ----------------------------> Backend
          username/password

Frontend <---------------------------- Backend
               JWT Token


          FUTURE API REQUESTS

Frontend ----------------------------> Backend
      Authorization: Bearer JWT

Frontend <---------------------------- Backend
            Protected Response

'''

#password bearer is used tto extract the token from the request header
oauth2_schema = OAuth2PasswordBearer(tokenUrl='token')

pwd_context = CryptContext(schemes=["bcrypt"],deprecated='auto')

#Secretkeyforencoding/decodingJWT
SECRET_KEY= "mysecretkey"
ALGORITHM ="HS256"
ACCESS_TOKEN_EXPIRE_MINUTES =30

